In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

In [2]:
class Critic(nn.Module):
  def __init__(self,imgc,feat):
    super().__init__()
    self.crit=nn.Sequential(
        nn.Conv2d(imgc,feat,kernel_size=4,stride=2,padding=1),
        nn.LeakyReLU(0.2),
        self._block(feat,feat*2,4,2,1),
        self._block(feat*2,feat*4,4,2,1),
        self._block(feat*4,feat*8,4,2,1),
        nn.Conv2d(feat*8,1,kernel_size=4,stride=2,padding=0)
    )
  def _block(self,inn,out,kernel,stride,pad):
    return nn.Sequential(
        nn.Conv2d(inn,out,kernel,stride,pad,bias=False),
        nn.InstanceNorm2d(out,affine=True),
        nn.LeakyReLU(0.2)
    )
  def forward(self,x):
    return self.crit(x)

In [3]:
class Generator(nn.Module):
  def __init__(self,noi,img,feat):
    super().__init__()
    self.gen=nn.Sequential(
        self._block(noi,feat*16,4,1,0),
        self._block(feat*16,feat*8,4,2,1),
        self._block(feat*8,feat*4,4,2,1),
        self._block(feat*4,feat*2,4,2,1),
        nn.ConvTranspose2d(feat*2,img,kernel_size=4,stride=2,padding=1),
        nn.Tanh()
    )
  def _block(self,inn,out,kern,stri,pad):
    return nn.Sequential(
        nn.ConvTranspose2d(inn,out,kern,stri,pad,bias=False),
        nn.BatchNorm2d(out),
        nn.ReLU()
    )
  def forward(self,x):
    return self.gen(x)

In [4]:
def initalise(model):
  for m in model.modules():
    if isinstance(m,(nn.Conv2d,nn.ConvTranspose2d,nn.BatchNorm2d)):
      nn.init.normal_(m.weight.data,0.0,0.02)

In [5]:
def grad_pen(cri,real,fake,device='cpu'):
  batch,c,h,w,=real.shape
  alpha=torch.rand((batch,1,1,1)).repeat(1,c,h,w).to(device)
  inter=real*alpha+fake*(1-alpha)
  mix=cri(inter)
  gradi=torch.autograd.grad(
      inputs=inter,
      outputs=mix,
      grad_outputs=torch.ones_like(mix),
      create_graph=True,
      retain_graph=True
  )[0]
  gradi=gradi.view(gradi.shape[0],-1)
  gradi_norm=gradi.norm(2,dim=1)
  gradi_pen=torch.mean((gradi_norm-1)**2)
  return gradi_pen

In [6]:
device='cuda' if torch.cuda.is_available() else 'cpu'
lr=1e-4
batch=64
img_size=64
chnl=1
z_dim=100
epochs=100
cfeat=16
gfeat=16
cri_it=5
lambdag=10

In [7]:
transforms=transforms.Compose(
    [
        transforms.Resize(img_size),
        transforms.ToTensor(),
        transforms.Normalize([0.5 for _ in range(chnl)],[0.5 for _ in range(chnl)])
    ]
)

In [8]:
data=datasets.MNIST(root='./dataset',transform=transforms,download=True)

100%|██████████| 9.91M/9.91M [00:02<00:00, 4.94MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 131kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.20MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 12.5MB/s]


In [9]:
loader=DataLoader(
    data,batch_size=batch,shuffle=True
)

In [10]:
gen=Generator(z_dim,chnl,gfeat).to(device)
critic=Critic(chnl,cfeat).to(device)

In [11]:
initalise(gen)
initalise(critic)

In [12]:
gopt=optim.Adam(gen.parameters(),lr=lr,betas=(0.0,0.9))
copt=optim.Adam(critic.parameters(),lr=lr,betas=(0.0,0.9))

In [13]:
noi=torch.randn(32,z_dim,1,1).to(device)
writer=SummaryWriter(f'logs/wgan_gp')
step=0

In [14]:
for epoch in range(epochs):
  for idx,(img,_) in enumerate(tqdm(loader)):
    img=img.to(device)
    cbs=img.shape[0]
    for _ in range(cri_it):
      noise=torch.randn(cbs,z_dim,1,1).to(device)
      fake=gen(noise)
      creal=critic(img).reshape(-1)
      cfake=critic(fake).reshape(-1)
      gp=grad_pen(critic,img,fake,device=device)
      lcritic=(-(torch.mean(creal)-torch.mean(cfake))+lambdag*gp)
      critic.zero_grad()
      lcritic.backward(retain_graph=True)
      copt.step()
    fgen=critic(fake).reshape(-1)
    lgen=-torch.mean(fgen)
    gen.zero_grad()
    lgen.backward()
    gopt.step()
    if idx % 100 == 0 and idx > 0:
      print(
          f"Epoch [{epoch}] Batch {idx}/{len(loader)} \
          Loss D: {lcritic:.4f}, loss G: {lgen:.4f}")
      with torch.no_grad():
        fake = gen(noi)
        img_grid_real = torchvision.utils.make_grid(img[:32], normalize=True)
        img_grid_fake = torchvision.utils.make_grid(fake[:32], normalize=True)
        writer.add_image("Real", img_grid_real, global_step=step)
        writer.add_image("Fake", img_grid_fake, global_step=step)
        step+=1

 11%|█         | 100/938 [00:16<01:59,  6.98it/s]

Epoch [0] Batch 100/938           Loss D: -120.0460, loss G: 65.5135


 21%|██▏       | 201/938 [00:33<02:04,  5.93it/s]

Epoch [0] Batch 200/938           Loss D: -153.3029, loss G: 100.1884


 32%|███▏      | 301/938 [00:49<02:09,  4.93it/s]

...

Epoch [2] Batch 100/938           Loss D: -10.5338, loss G: 83.2215


 21%|██▏       | 201/938 [00:29<02:01,  6.05it/s]

Epoch [2] Batch 200/938           Loss D: -9.6345, loss G: 92.2022


 29%|██▉       | 271/938 [00:39<01:36,  6.88it/s]


KeyboardInterrupt: 

In [ ]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir logs